# CAPTOR on natural images

A frozen DINOv2 ViT-B/14 (with registers) read at 196 pixels, and a table counted
from the captions of 15,000 COCO training images. No text encoder is used at any
point: each term is a row of the table, and the score of a term at a patch is
read from the visual words the patch falls into.

The table (about 67 MB) and the DINOv2 weights are downloaded on first use.
Everything runs on the CPU.

In [ ]:
# Setup for Google Colab: fetch the code and the example images (skipped when run locally)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/captor-repo"):
        !git clone -q https://github.com/EladYoshai/captor.git /content/captor-repo
        !pip install -q -e /content/captor-repo
    %cd /content/captor-repo/examples
    sys.path.insert(0, "/content/captor-repo")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from captor import CAPTOR
from captor.viz import crop_like_encoder, heatmap, overlay

model = CAPTOR.from_pretrained("coco_dinov2reg_k640")
print(len(model.terms), "terms:", ", ".join(model.terms[:12]), "...")

## Name every patch with one of a few terms

In [ ]:
image = Image.open(Path("images") / "cat_lizard.jpg")
terms = ["cat", "dog", "bird", "bench", "potted plant"]
seg = model.segment(image, terms)
print({t: round(s, 2) for t, s in seg.shares().items()})

view = crop_like_encoder(image, center_crop=True)   # DINOv2 reads the centre square
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(view); ax[1].imshow(overlay(view, seg.labels))
for a in ax: a.axis("off")

## One row of the score matrix

`seg.scores[..., i]` is the score of term `i` at every patch, the quantity a
cross-attention layer would pool over.

In [ ]:
fig, ax = plt.subplots(1, len(terms), figsize=(3.2 * len(terms), 3.4))
for i, t in enumerate(terms):
    ax[i].imshow(Image.blend(view.convert("RGB"), heatmap(seg.scores[..., i], view.size), 0.6))
    ax[i].set_title(t); ax[i].axis("off")

## Ranking whole images (Eq. score)

In [ ]:
for name in ["cat_lizard.jpg", "girl_dog.jpg", "pizza.jpg", "horse.jpg"]:
    s = model.score(Path("images") / name, ["cat", "dog", "pizza", "horse", "person"])
    print(f"{name:<16}", "  ".join(f"{t} {v:+6.1f}" for t, v in s.items()))